In [1]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from backend.preprocess import Preprocessor
from backend.datasets import create_dataloaders

In [2]:
pre = Preprocessor(csv_path="../datasets/Train.csv", images_dir="../datasets/Images")
train, val, report = pre.split(return_report=True)

for key, value in report.items():
    print(f"{key}: {value}")

Removed 0 duplicate rows by img_IDS.
[WARNING] 1 row(s) have no matching image files — dropped.
rows_raw: 6249
removed_duplicate_ids: 0
rows_after_dedup: 6249
rows_with_files: 6248
rows_missing_files: 1
label_distribution_raw: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'Love': 694, 'You': 694, 'Friend': 694}
label_distribution_clean: {'Enough/Satisfied': 695, 'Mosque': 695, 'Seat': 695, 'Temple': 694, 'Church': 694, 'Me': 694, 'You': 694, 'Friend': 694, 'Love': 693}


Для текущей задачи распознавания жестов оптимально начать с базового пайплайна аугментаций: `Resize` + `Normalize`.  
Далее стоит добавить мягкие геометрические преобразования (`ShiftScaleRotate` с небольшими пределами), чтобы повысить устойчивость модели к сдвигам и изменению масштаба.  
Также полезны слабые фотометрические аугментации (`RandomBrightnessContrast`, `GaussianNoise`) с низкой вероятностью применения.

Агрессивные искажения (сильный blur, большие вырезания, чрезмерные деформации) лучше не использовать на старте, так как они могут ухудшить распознавание семантики жестов.  
`HorizontalFlip` нужно применять с осторожностью: для языка жестов отражение может менять смысл знака и потенциально вносить шум в разметку.  

Если в данных заметен дисбаланс классов, рекомендуется дополнительно использовать `class-balanced sampling` и более активные аугментации именно для минорных классов.

In [3]:
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32

train_transform = A.Compose([
    A.Resize(*IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(*IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_loader, val_loader, label_to_idx = create_dataloaders(
    train_df=train,
    val_df=val,
    images_dir="../datasets/Images",
    image_size=IMAGE_SIZE,
    train_transform=train_transform,
    val_transform=val_transform,
    batch_size=BATCH_SIZE,
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Num classes: {len(label_to_idx)}")
print(f"Label mapping: {label_to_idx}")

Train batches: 130, Val batches: 65
Num classes: 9
Label mapping: {'Church': 0, 'Enough/Satisfied': 1, 'Friend': 2, 'Love': 3, 'Me': 4, 'Mosque': 5, 'Seat': 6, 'Temple': 7, 'You': 8}
